# OLX Carros Scraper (Playwright)

Este notebook faz scraping do OLX (`Carros`) por **marca -> modelo -> todas as páginas**.

Ele inclui:
- Descoberta inicial de marcas (a partir do site).
- Lista editável de marcas para ligares/desligares facilmente.
- Gravação incremental em `JSONL` e `CSV`.
- Checkpoint para retomar em caso de paragem/interrupção.
- Export final deduplicado para `CSV` e `JSON`.


In [1]:
# Célula opcional: instala dependências se ainda não tiveres no ambiente atual.
%pip install -U playwright pandas
# !playwright install chromium


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importações principais usadas no scraping e na escrita dos ficheiros.
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin
import asyncio
import csv
import json
import re

import pandas as pd
from playwright.async_api import async_playwright

# URL base da categoria de carros no OLX.
BASE_URL = "https://www.olx.pt/carros-motos-e-barcos/carros/"

# Pasta de output e nomes dos ficheiros gerados.
OUTPUT_DIR = Path("output")
INCREMENTAL_JSONL_FILE = OUTPUT_DIR / "olx_carros_incremental.jsonl"
INCREMENTAL_CSV_FILE = OUTPUT_DIR / "olx_carros_incremental.csv"
FINAL_CSV_FILE = OUTPUT_DIR / "olx_carros_final.csv"
FINAL_JSON_FILE = OUTPUT_DIR / "olx_carros_final.json"
CHECKPOINT_FILE = OUTPUT_DIR / "checkpoint_olx_carros.json"
BRANDS_CACHE_FILE = OUTPUT_DIR / "brands_discovered.json"

# Campos padronizados (para CSV e JSONL).
FIELDNAMES = [
    "ad_id",
    "title",
    "price",
    "location_and_date",
    "specs",
    "ad_url",
    "brand_selected",
    "model_selected",
    "source_page_url",
    "scraped_at_utc",
]

# Ajustes simples de execução.
HEADLESS = True
REQUEST_DELAY_SECONDS = 1.0


In [3]:
# Funções auxiliares para manter o código principal simples e legível.

def ensure_output_dir() -> None:
    """Cria a pasta de output caso ainda não exista."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def utc_now_iso() -> str:
    """Devolve timestamp UTC em formato ISO para rastrear a extração."""
    return datetime.now(timezone.utc).isoformat()


def clean_text(value: str) -> str:
    """Normaliza espaços em branco em qualquer texto extraído."""
    if value is None:
        return ""
    return re.sub(r"\s+", " ", value).strip()


async def cookie_overlay_visible(page) -> bool:
    """Verifica se o overlay de cookies da OneTrust está a bloquear cliques."""
    try:
        return await page.evaluate(
            """() => {
                const selectors = [
                    '#onetrust-consent-sdk .onetrust-pc-dark-filter',
                    '#onetrust-consent-sdk #onetrust-pc-sdk',
                    '#onetrust-consent-sdk #onetrust-banner-sdk',
                    '.onetrust-pc-dark-filter',
                ];
                return selectors.some((selector) => {
                    return [...document.querySelectorAll(selector)].some((el) => {
                        const st = window.getComputedStyle(el);
                        if (!st) return false;
                        const hidden = st.display === 'none' || st.visibility === 'hidden' || st.opacity === '0';
                        const rect = el.getBoundingClientRect();
                        return !hidden && rect.width > 0 && rect.height > 0;
                    });
                });
            }"""
        )
    except Exception:
        return False


async def hide_cookie_overlay(page) -> None:
    """Fallback: esconde o overlay quando ele continua a bloquear interação."""
    try:
        await page.evaluate(
            """() => {
                const selectors = [
                    '#onetrust-consent-sdk .onetrust-pc-dark-filter',
                    '#onetrust-consent-sdk #onetrust-pc-sdk',
                    '#onetrust-consent-sdk #onetrust-banner-sdk',
                    '.onetrust-pc-dark-filter',
                    '#onetrust-consent-sdk',
                ];
                for (const selector of selectors) {
                    for (const el of document.querySelectorAll(selector)) {
                        el.style.setProperty('display', 'none', 'important');
                        el.style.setProperty('visibility', 'hidden', 'important');
                        el.style.setProperty('pointer-events', 'none', 'important');
                    }
                }
            }"""
        )
    except Exception:
        pass


async def accept_cookies(page) -> None:
    """Aceita/fecha cookies e remove overlays que possam bloquear os cliques."""
    selectors = [
        "#onetrust-accept-btn-handler",
        "#accept-recommended-btn-handler",
        "button:has-text('Aceitar')",
        "button:has-text('Aceitar tudo')",
        "button:has-text('Accept')",
        "button:has-text('Accept all')",
        "button[aria-label='Fechar']",
        ".onetrust-close-btn-handler",
    ]

    for _ in range(3):
        clicked_any = False
        for selector in selectors:
            locator = page.locator(selector)
            if await locator.count() > 0:
                try:
                    await locator.first.click(timeout=1800, force=True)
                    await page.wait_for_timeout(300)
                    clicked_any = True
                except Exception:
                    pass

        if not await cookie_overlay_visible(page):
            return

        if not clicked_any:
            await page.wait_for_timeout(300)

    await hide_cookie_overlay(page)


async def click_with_retry(page, locator, timeout: int = 15000) -> None:
    """Tenta clicar normalmente; se houver bloqueio de overlay, faz fallback."""
    try:
        await locator.click(timeout=timeout)
        return
    except Exception:
        await accept_cookies(page)
        await hide_cookie_overlay(page)
        await page.wait_for_timeout(250)
        await locator.click(timeout=timeout, force=True)


def load_checkpoint(path: Path) -> dict:
    """Lê o checkpoint atual (ou cria estrutura vazia)."""
    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "completed_models": [],
        "in_progress": None,
        "total_rows_written": 0,
        "updated_at_utc": None,
    }


def save_checkpoint(path: Path, payload: dict) -> None:
    """Grava checkpoint em disco para retomar a qualquer momento."""
    payload["updated_at_utc"] = utc_now_iso()
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def append_jsonl(rows: list, path: Path) -> None:
    """Acrescenta linhas no JSONL incremental."""
    if not rows:
        return
    with path.open("a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def append_csv(rows: list, path: Path, fieldnames: list) -> None:
    """Acrescenta linhas no CSV incremental (com cabeçalho na primeira escrita)."""
    if not rows:
        return
    file_exists = path.exists() and path.stat().st_size > 0
    with path.open("a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)


def model_key(brand_name: str, model_label: str) -> str:
    """Chave única por combinação marca-modelo para checkpoint."""
    return f"{brand_name}|||{model_label}"


async def list_brands_from_home(page) -> list:
    """
    Vai à página base e devolve as marcas disponíveis no dropdown de Subcategoria.
    Cada item devolvido: {name, count}.
    """
    await page.goto(BASE_URL, wait_until="domcontentloaded", timeout=120000)
    await page.wait_for_selector("[data-testid='category-dropdown']", timeout=45000)
    await page.wait_for_timeout(1200)
    await accept_cookies(page)

    dropdowns = page.locator("[data-testid='category-dropdown']")
    if await dropdowns.count() < 2:
        raise RuntimeError("Não foi possível encontrar o dropdown de marcas.")

    await click_with_retry(page, dropdowns.nth(1), timeout=20000)
    await page.wait_for_timeout(900)
    await page.wait_for_selector("#category-dropdown-list", timeout=10000)

    raw_values = await page.locator("#category-dropdown-list button").all_inner_texts()

    brands = []
    for raw in raw_values:
        text = clean_text(raw)
        if not text:
            continue
        if text.lower().startswith("todos os anúncios"):
            continue

        match = re.match(r"^(.*?)(\d+)?$", text)
        if match:
            name = clean_text(match.group(1))
            count = int(match.group(2)) if match.group(2) else None
        else:
            name = text
            count = None

        if name:
            brands.append({"name": name, "count": count})

    await page.keyboard.press("Escape")
    return brands


async def open_multi_select_filter(page, filter_label: str) -> bool:
    """Abre um filtro do tipo multi-select (ex.: Modelo) pelo texto do label."""
    return await page.evaluate(
        """(targetLabel) => {
            const blocks = [...document.querySelectorAll('[data-testid=\\"multi-select-filter\\"]')];
            const target = blocks.find((block) => {
                const label = block.querySelector('.css-95hdyi');
                const text = (label?.textContent || '').trim();
                return text === targetLabel;
            });
            if (!target) return false;
            const button = target.querySelector('[data-testid=\\"dropdown-head\\"]');
            if (!button) return false;
            button.click();
            return true;
        }""",
        filter_label,
    )


async def get_brand_url_and_models(page, brand_name: str) -> tuple[str, list]:
    """
    Entra numa marca e devolve:
    - brand_url: URL da marca (ex.: .../audi/)
    - models: lista de labels de modelos dessa marca
    """
    await page.goto(BASE_URL, wait_until="domcontentloaded", timeout=120000)
    await page.wait_for_selector("[data-testid='category-dropdown']", timeout=45000)
    await page.wait_for_timeout(1200)
    await accept_cookies(page)

    dropdowns = page.locator("[data-testid='category-dropdown']")
    if await dropdowns.count() < 2:
        raise RuntimeError("Dropdown de marcas não encontrado.")

    await click_with_retry(page, dropdowns.nth(1), timeout=20000)
    await page.wait_for_timeout(900)

    selected = await page.evaluate(
        """(targetBrand) => {
            const cleanName = (text) => {
                const raw = (text || '').replace(/\\s+/g, ' ').trim();
                return raw.replace(/\\d+$/, '').trim();
            };
            const options = [...document.querySelectorAll('#category-dropdown-list button')];
            const target = options.find((btn) => cleanName(btn.textContent) === targetBrand);
            if (!target) return false;
            target.click();
            return true;
        }""",
        brand_name,
    )
    if not selected:
        return "", []

    await page.wait_for_load_state("domcontentloaded")
    await page.wait_for_timeout(1200)
    await accept_cookies(page)
    brand_url = page.url

    model_filter_found = await open_multi_select_filter(page, "Modelo")
    if not model_filter_found:
        return brand_url, []

    await page.wait_for_timeout(700)

    models = await page.evaluate(
        """() => {
            const values = [];
            const options = [...document.querySelectorAll('label[role=\\"option\\"]')];
            for (const option of options) {
                const text = (option.textContent || '').replace(/\\s+/g, ' ').trim();
                if (!text) continue;
                if (text.toLowerCase() === 'mostrar tudo') continue;
                values.push(text);
            }
            return values;
        }"""
    )

    await page.keyboard.press("Escape")
    return brand_url, models


async def click_model_and_get_url(page, brand_url: str, model_label: str) -> str | None:
    """
    Clica num modelo específico e devolve a URL filtrada desse modelo.
    Isto evita adivinhar slugs de modelo.
    """
    await page.goto(brand_url, wait_until="domcontentloaded", timeout=120000)
    await page.wait_for_timeout(1200)
    await accept_cookies(page)

    model_filter_found = await open_multi_select_filter(page, "Modelo")
    if not model_filter_found:
        return None

    await page.wait_for_timeout(600)
    old_url = page.url

    clicked = await page.evaluate(
        """(targetModel) => {
            const options = [...document.querySelectorAll('label[role=\"option\"]')];
            const target = options.find((option) => {
                const text = (option.textContent || '').replace(/\\s+/g, ' ').trim();
                return text === targetModel;
            });
            if (!target) return false;
            target.click();
            return true;
        }""",
        model_label,
    )

    if not clicked:
        return None

    # Alguns modelos só aplicam filtro quando o dropdown fecha.
    for _ in range(6):
        await page.wait_for_timeout(400)
        if page.url != old_url and "filter_enum_modelo" in page.url:
            return page.url

    try:
        await page.keyboard.press("Escape")
    except Exception:
        pass

    for _ in range(10):
        await page.wait_for_timeout(400)
        if page.url != old_url and "filter_enum_modelo" in page.url:
            return page.url

    # Fallback: clicar fora do dropdown para confirmar seleção.
    try:
        await page.mouse.click(20, 20)
    except Exception:
        pass

    for _ in range(6):
        await page.wait_for_timeout(400)
        if page.url != old_url and "filter_enum_modelo" in page.url:
            return page.url

    return None


async def extract_cards(page, brand_name: str, model_label: str) -> list:
    """Extrai os dados dos cards da página atual."""
    payload = {
        "brand": brand_name,
        "model": model_label,
        "source_page_url": page.url,
        "scraped_at_utc": utc_now_iso(),
    }

    rows = await page.evaluate(
        """(data) => {
            const clean = (value) => (value || '').replace(/\\s+/g, ' ').trim();
            const cards = [...document.querySelectorAll('[data-cy=\\"l-card\\"]')];

            return cards.map((card) => {
                const title = clean(card.querySelector('[data-cy=\\"ad-card-title\\"] h4')?.textContent);
                const price = clean(card.querySelector('[data-testid=\\"ad-price\\"]')?.textContent);
                const locationAndDate = clean(card.querySelector('[data-testid=\\"location-date\\"]')?.textContent);

                const specs = [...card.querySelectorAll('.css-1kfqt7f .css-h59g4b')]
                    .map((el) => clean(el.textContent))
                    .filter(Boolean)
                    .join(' | ');

                const adAnchor = card.querySelector('[data-cy=\\"ad-card-title\\"] a') || card.querySelector('a[href]');
                const href = adAnchor?.getAttribute('href') || '';
                const absoluteUrl = href ? new URL(href, location.origin).href : '';
                const normalizedUrl = absoluteUrl ? absoluteUrl.split('?')[0] : '';

                return {
                    ad_id: clean(card.id || ''),
                    title,
                    price,
                    location_and_date: locationAndDate,
                    specs,
                    ad_url: normalizedUrl,
                    brand_selected: data.brand,
                    model_selected: data.model,
                    source_page_url: data.source_page_url,
                    scraped_at_utc: data.scraped_at_utc,
                };
            });
        }""",
        payload,
    )

    return rows


async def get_next_page_url(page) -> str | None:
    """Devolve URL da próxima página da paginação (ou None se for a última)."""
    forward = page.locator("[data-cy='pagination-forward']")
    if await forward.count() == 0:
        return None
    href = await forward.first.get_attribute("href")
    if not href:
        return None
    return urljoin("https://www.olx.pt", href)


In [4]:
# Descobre as marcas no site (1.ª visita) e guarda essa fotografia localmente.
ensure_output_dir()

async def discover_brands_once():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)
        page = await browser.new_page(viewport={"width": 1600, "height": 2200})

        discovered = await list_brands_from_home(page)
        await browser.close()
        return discovered

discovered_brands = await discover_brands_once()
ALL_BRANDS = [item["name"] for item in discovered_brands]

with BRANDS_CACHE_FILE.open("w", encoding="utf-8") as f:
    json.dump(discovered_brands, f, ensure_ascii=False, indent=2)

print(f"Total de marcas descobertas: {len(discovered_brands)}")
for item in discovered_brands:
    print(f"- {item['name']} (anúncios aproximados: {item['count']})")

print(f"\nLista guardada em: {BRANDS_CACHE_FILE}")


Total de marcas descobertas: 114
- Abarth (anúncios aproximados: 103)
- AC (anúncios aproximados: None)
- Acura (anúncios aproximados: 1)
- Aiways (anúncios aproximados: 3)
- Aixam (anúncios aproximados: 14)
- Alfa Romeo (anúncios aproximados: 392)
- Alpina (anúncios aproximados: None)
- Alpine (anúncios aproximados: 5)
- Amphicar (anúncios aproximados: None)
- Aston Martin (anúncios aproximados: 18)
- Audi (anúncios aproximados: 2206)
- Austin (anúncios aproximados: 8)
- Austin Healey (anúncios aproximados: None)
- Austin Morris (anúncios aproximados: 4)
- Autobianchi (anúncios aproximados: 1)
- Bedford (anúncios aproximados: 2)
- Bellier (anúncios aproximados: 1)
- Bentley (anúncios aproximados: 38)
- BMW (anúncios aproximados: 5044)
- Brabus (anúncios aproximados: None)
- Bugatti (anúncios aproximados: None)
- Buick (anúncios aproximados: 1)
- BYD (anúncios aproximados: 120)
- Cadillac (anúncios aproximados: 1)
- Casalini (anúncios aproximados: None)
- Caterham (anúncios aproximados

In [5]:
# Lista editável de marcas a processar.
# Comentário rápido: comenta linhas que NÃO queres usar.
# Se preferires usar exatamente o que o site devolveu na célula anterior, usa:
# BRANDS_TO_SCRAPE = ALL_BRANDS.copy()

BRANDS_TO_SCRAPE = [
    "Abarth",
    #"AC",
    "Acura",
    #"Aiways",
    #"Aixam",
    "Alfa Romeo",
    #"Alpina",
    "Alpine",
    #"Amphicar",
    "Aston Martin",
    "Audi",
    "Austin",
    "Austin Healey",
    "Austin Morris",
    #"Autobianchi",
    #"Bedford",
    #"Bellier",
    #"Bentley",
    "BMW",
    #"Brabus",
    #"Bugatti",
    #"Buick",
    "BYD",
    "Cadillac",
    #"Casalini",
    #"Caterham",
    #"Chatenet",
    "Chevrolet",
    "Chrysler",
    "Citroën",
    "Cupra",
    "Dacia",
    "Daewoo",
    "Daihatsu",
    "Daimler",
    #"Datsun",
    #"Devinci",
    #"Dodge",
    "DS",
    "e.GO",
    #"Ferrari",
    "Fiat",
    #"Fisker",
    "Ford",
    "GMC",
    "Genesis",
    "Hillman",
    "Honda",
    "Hummer",
    "Hyundai",
    "Infiniti",
    "Isuzu",
    "Iveco",
    "Jaguar",
    "Jeep",
    "JaguarSport",
    "Jiayuan",
    "JDM",
    "Kaiser",
    "Kia",
    #"Lada",
    #"Lamborghini",
    "Lancia",
    "Land Rover",
    "Lexus",
    "Ligier",
    "Lotus",
    "Man",
    "Lincoln",
    #"Maserati",
    "Maybach",
    "Mazda",
    #"Mclaren",
    "Mercedes-Benz",
    "MG",
    #"Microcar",
    "MINI",
    "Meyers Manx",
    "Mitsubishi",
    "Morgan",
    "Morris",
    "Nash",
    "Nissan",
    #"NSU",
    "Opel",
    "Outra não listada",
    "Peugeot",
    "Plymouth",
    "Polestar",
    "Porsche",
    #"Pontiac",
    "Renault",
    #"Riley",
    #"Rolls Royce",
    "Rover",
    "Saab",
    "Seat",
    #"Shelby",
    "Skoda",
    "Smart",
    "SsangYong",
    "Subaru",
    "Suzuki",
    "Tata",
    "Tesla",
    "Toyota",
    "Triumph",
    #"UMM",
    "Today Sunshine",
    "Vanderhall",
    "Vauxhall",
    "VW",
    "Wiesmann",
    "Volvo",
]

print(f"Marcas selecionadas para scraping: {len(BRANDS_TO_SCRAPE)}")


Marcas selecionadas para scraping: 83


In [6]:
# Execução principal do scraping com checkpoint e gravação incremental.
ensure_output_dir()
checkpoint = load_checkpoint(CHECKPOINT_FILE)
completed_models = set(checkpoint.get("completed_models", []))

print("Checkpoint carregado:")
print(f"- Modelos já concluídos: {len(completed_models)}")
print(f"- Linhas já escritas: {checkpoint.get('total_rows_written', 0)}")
print(f"- In progress: {checkpoint.get('in_progress')}")

async def run_scraping():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)
        context = await browser.new_context(
            locale="pt-PT",
            viewport={"width": 1600, "height": 2200},
        )
        page = await context.new_page()

        # Valida marcas selecionadas contra as marcas ativas no site neste momento.
        discovered_brands = await list_brands_from_home(page)
        discovered_names = {item['name'] for item in discovered_brands}

        selected_brands = [brand for brand in BRANDS_TO_SCRAPE if brand in discovered_names]
        missing_brands = [brand for brand in BRANDS_TO_SCRAPE if brand not in discovered_names]

        if missing_brands:
            print("")
            print("Marcas ignoradas (não encontradas agora no site):")
            for brand in missing_brands:
                print(f"- {brand}")

        print("")
        print(f"Marcas válidas para scraping nesta execução: {len(selected_brands)}")

        for brand_name in selected_brands:
            print("")
            print(f"========== Marca: {brand_name} ==========")

            try:
                brand_url, models = await get_brand_url_and_models(page, brand_name)
            except Exception as error:
                print(f"[ERRO] Falha ao carregar marca '{brand_name}': {error}")
                continue

            if not brand_url:
                print(f"[AVISO] Não foi possível entrar na marca '{brand_name}'.")
                continue

            if not models:
                print(f"[AVISO] Sem modelos disponíveis para '{brand_name}'.")
                continue

            print(f"Modelos encontrados: {len(models)}")

            for model_label in models:
                key = model_key(brand_name, model_label)
                if key in completed_models:
                    continue

                in_progress = checkpoint.get("in_progress") or {}

                # Se já estava em progresso, retoma da URL da última página pendente.
                if (
                    in_progress.get("brand_name") == brand_name
                    and in_progress.get("model_label") == model_label
                    and in_progress.get("next_page_url")
                ):
                    current_url = in_progress["next_page_url"]
                    print(f"[RETOMA] {brand_name} -> {model_label}")
                else:
                    # Obtém URL exata do modelo clicando na opção (evita erros de slug).
                    model_url = await click_model_and_get_url(page, brand_url, model_label)
                    if not model_url:
                        print(f"[AVISO] Modelo não clicável: {brand_name} -> {model_label}")
                        continue
                    current_url = model_url

                # Guarda checkpoint antes de começar a paginar este modelo.
                checkpoint["in_progress"] = {
                    "brand_name": brand_name,
                    "model_label": model_label,
                    "next_page_url": current_url,
                }
                checkpoint["completed_models"] = sorted(completed_models)
                save_checkpoint(CHECKPOINT_FILE, checkpoint)

                visited_pages = set()

                while current_url and current_url not in visited_pages:
                    visited_pages.add(current_url)
                    print(f"  [PÁGINA] {current_url}")

                    await page.goto(current_url, wait_until="domcontentloaded", timeout=120000)
                    await page.wait_for_timeout(1200)
                    await accept_cookies(page)

                    rows = await extract_cards(page, brand_name, model_label)
                    append_jsonl(rows, INCREMENTAL_JSONL_FILE)
                    append_csv(rows, INCREMENTAL_CSV_FILE, FIELDNAMES)

                    checkpoint["total_rows_written"] = checkpoint.get("total_rows_written", 0) + len(rows)
                    next_url = await get_next_page_url(page)

                    # Atualiza checkpoint a cada página para poder retomar no ponto certo.
                    checkpoint["in_progress"] = {
                        "brand_name": brand_name,
                        "model_label": model_label,
                        "next_page_url": next_url,
                    }
                    checkpoint["completed_models"] = sorted(completed_models)
                    save_checkpoint(CHECKPOINT_FILE, checkpoint)

                    if next_url == current_url:
                        break

                    current_url = next_url
                    await asyncio.sleep(REQUEST_DELAY_SECONDS)

                # Marca modelo como concluído quando acaba a paginação.
                completed_models.add(key)
                checkpoint["completed_models"] = sorted(completed_models)
                checkpoint["in_progress"] = None
                save_checkpoint(CHECKPOINT_FILE, checkpoint)

        await context.close()
        await browser.close()

await run_scraping()

print("\nScraping concluído para as marcas/modelos selecionados.")
print(f"JSONL incremental: {INCREMENTAL_JSONL_FILE}")
print(f"CSV incremental:   {INCREMENTAL_CSV_FILE}")
print(f"Checkpoint:        {CHECKPOINT_FILE}")


Checkpoint carregado:
- Modelos já concluídos: 0
- Linhas já escritas: 0
- In progress: None

Marcas válidas para scraping nesta execução: 83

========== Marca: Abarth ==========
Modelos encontrados: 11
  [PÁGINA] https://www.olx.pt/carros-motos-e-barcos/carros/abarth/?search%5Bfilter_enum_modelo%5D%5B0%5D=124-spider
  [PÁGINA] https://www.olx.pt/carros-motos-e-barcos/carros/abarth/?search%5Bfilter_enum_modelo%5D%5B0%5D=500
  [PÁGINA] https://www.olx.pt/carros-motos-e-barcos/carros/abarth/?search%5Bfilter_enum_modelo%5D%5B0%5D=500c
  [PÁGINA] https://www.olx.pt/carros-motos-e-barcos/carros/abarth/?search%5Bfilter_enum_modelo%5D%5B0%5D=500e
  [PÁGINA] https://www.olx.pt/carros-motos-e-barcos/carros/abarth/?search%5Bfilter_enum_modelo%5D%5B0%5D=500e-c
  [PÁGINA] https://www.olx.pt/carros-motos-e-barcos/carros/abarth/?search%5Bfilter_enum_modelo%5D%5B0%5D=595
  [PÁGINA] https://www.olx.pt/carros-motos-e-barcos/carros/abarth/?page=2&search%5Bfilter_enum_modelo%5D%5B0%5D=595
  [PÁGINA] http

CancelledError: 

In [ ]:
# Consolida o incremental e gera outputs finais deduplicados (CSV + JSON).
if not INCREMENTAL_JSONL_FILE.exists():
    print("Não existe JSONL incremental. Corre primeiro a célula de scraping.")
else:
    rows = []
    with INCREMENTAL_JSONL_FILE.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))

    df = pd.DataFrame(rows)
    before = len(df)

    # Deduplicação simples para evitar repetição em retomas de checkpoint.
    dedupe_cols = ["ad_id", "ad_url", "brand_selected", "model_selected"]
    existing_cols = [col for col in dedupe_cols if col in df.columns]
    if existing_cols:
        df = df.drop_duplicates(subset=existing_cols, keep="last")

    after = len(df)

    df.to_csv(FINAL_CSV_FILE, index=False, encoding="utf-8-sig")
    df.to_json(FINAL_JSON_FILE, orient="records", force_ascii=False, indent=2)

    print(f"Registos no incremental (JSONL): {before}")
    print(f"Registos após deduplicação:      {after}")
    print(f"CSV final:  {FINAL_CSV_FILE}")
    print(f"JSON final: {FINAL_JSON_FILE}")


In [ ]:
# Pré-visualização rápida do CSV final (se já existir).
if FINAL_CSV_FILE.exists():
    preview_df = pd.read_csv(FINAL_CSV_FILE)
    print(f"Total linhas no CSV final: {len(preview_df)}")
    display(preview_df.head(10))
else:
    print("CSV final ainda não existe.")
